# 3단계 교차검증과 보상 인벤토리

이 노트북은 2단계 중간 산출물을 읽어 종류별 뷰와 날짜별 뷰를 교차검증하고, `data/interim/reward_inventory.csv`를 저장한다. 최종 processed 산출물은 만들지 않는다.


In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

interim_dir = project_root / 'data' / 'interim'
TODAY = pd.Timestamp('2026-06-10')

In [2]:
parsed_by_type = pd.read_csv(interim_dir / 'parsed_by_type.csv', encoding='utf-8')
parsed_by_date = pd.read_csv(interim_dir / 'parsed_by_date.csv', encoding='utf-8')
tidy_prenorm = pd.read_csv(interim_dir / 'tidy_prenorm.csv', encoding='utf-8')

for df in [parsed_by_type, parsed_by_date, tidy_prenorm]:
    df['date'] = pd.to_datetime(df['date'])

print('parsed_by_type 행 수:', len(parsed_by_type))
print('parsed_by_date 행 수:', len(parsed_by_date))
print('tidy_prenorm 행 수:', len(tidy_prenorm))


parsed_by_type 행 수: 1140
parsed_by_date 행 수: 1160
tidy_prenorm 행 수: 1160


## 교차검증 A: 종류별 원문 주기


In [3]:
type_sorted = parsed_by_type.sort_values(['reward_raw', 'date']).copy()
type_sorted['actual_cycle_days'] = type_sorted.groupby('reward_raw')['date'].diff().dt.days
cycle_check = type_sorted[type_sorted['stated_cycle_days'].notna()].copy()
cycle_mismatch = cycle_check[
    cycle_check['actual_cycle_days'] != cycle_check['stated_cycle_days']
].copy()

print('주기 검증 대상 행 수:', len(cycle_check))
print('주기 불일치 행 수:', len(cycle_mismatch))
display(cycle_mismatch[[
    'reward_raw', 'date', 'stated_cycle_days', 'actual_cycle_days', 'trailing'
]].head(10))


주기 검증 대상 행 수: 1041
주기 불일치 행 수: 0


,reward_raw,date,stated_cycle_days,actual_cycle_days,trailing


## 교차검증 B: reward_raw별 등장 횟수 일치


In [4]:
by_type_freq = parsed_by_type.groupby('reward_raw').size().rename('freq_by_type')
by_date_rewards = parsed_by_date[
    (~parsed_by_date['no_event_week']) & parsed_by_date['reward_raw'].notna()
].copy()
by_date_freq = by_date_rewards.groupby('reward_raw').size().rename('freq_by_date')

freq_compare = pd.concat([by_type_freq, by_date_freq], axis=1).fillna(0).astype(int).reset_index()
freq_compare['is_exact_count_match'] = freq_compare['freq_by_type'] == freq_compare['freq_by_date']
matched = int(freq_compare['is_exact_count_match'].sum())
total = len(freq_compare)
match_rate = matched / total if total else 0

print('고유 reward_raw 전체 수:', total)
print('등장 횟수 일치 보상 수:', matched)
print('등장 횟수 불일치 보상 수:', total - matched)
print('등장 횟수 일치율:', match_rate)
display(freq_compare[~freq_compare['is_exact_count_match']]
        .sort_values(['freq_by_date', 'freq_by_type', 'reward_raw'], ascending=[False, False, True])
        .head(10))


고유 reward_raw 전체 수: 113
등장 횟수 일치 보상 수: 82
등장 횟수 불일치 보상 수: 31
등장 횟수 일치율: 0.7256637168141593


,reward_raw,freq_by_type,freq_by_date,is_exact_count_match
95,폴로와 프리토의 현상금 사냥 경험치 2배,53,31,False
112,"폴로와 프리토, 에스페시아의 현상금 사냥 경험치 2배",0,22,False
107,사냥 시 소환되는 엘리트 몬스터 수 증가,0,19,False
101,돌발미션 일일 클리어 가능 횟수 2배,0,4,False
104,룬 경험치 버프 지속시간 2배,0,3,False
75,익스트림 레벨업 타임,3,2,False
102,돌발미션 일일 클리어 가능 횟수 3배,0,2,False
17,데미지 스킨 선택 지급,2,1,False
99,강력한 환생의 불꽃 5개 사용 시마다 1개 지급,0,1,False
100,데미지 스킨 선택 지급(샤방샤방/샤방샤방(Ver.2)/샤방샤방(Ver.3)),0,1,False


## 보상 인벤토리


In [5]:
type_dates = parsed_by_type.groupby('reward_raw')['date'].agg(type_first='min', type_last='max')
date_dates = by_date_rewards.groupby('reward_raw')['date'].agg(date_first='min', date_last='max')
discontinued = parsed_by_type.groupby('reward_raw')['is_discontinued'].any().rename('is_discontinued')

inventory = freq_compare.set_index('reward_raw').join([type_dates, date_dates, discontinued])
inventory['first_date'] = inventory[['type_first', 'date_first']].min(axis=1)
inventory['last_date'] = inventory[['type_last', 'date_last']].max(axis=1)
inventory['is_discontinued'] = inventory['is_discontinued'].fillna(False).astype(bool)
inventory = inventory.reset_index()[[
    'reward_raw', 'freq_by_date', 'freq_by_type', 'first_date', 'last_date', 'is_discontinued'
]]
inventory = inventory.sort_values(
    ['freq_by_date', 'freq_by_type', 'reward_raw'], ascending=[False, False, True]
).reset_index(drop=True)
inventory['first_date'] = inventory['first_date'].dt.strftime('%Y-%m-%d')
inventory['last_date'] = inventory['last_date'].dt.strftime('%Y-%m-%d')
inventory.to_csv(interim_dir / 'reward_inventory.csv', index=False, encoding='utf-8')

print('reward_inventory 행 수:', len(inventory))
print('reward_inventory 단종 보상 수:', int(inventory['is_discontinued'].sum()))
print('reward_inventory 바이트:', (interim_dir / 'reward_inventory.csv').stat().st_size)
display(inventory.head(20))


reward_inventory 행 수: 113
reward_inventory 단종 보상 수: 71
reward_inventory 바이트: 9603


,reward_raw,freq_by_date,freq_by_type,first_date,last_date,is_discontinued
0,룬 경험치 버프 효과 +100%,87,87,2017-03-26,2026-06-07,False
1,콤보킬 경험치 획득량 +300%,86,86,2017-11-26,2026-06-07,False
2,스타포스 강화 비용 30% 할인,73,73,2017-04-23,2026-05-31,False
3,어빌리티 재설정 비용 50% 할인,69,69,2017-04-09,2026-05-17,False
4,주문의 흔적 강화 비용 50% 할인,68,68,2018-03-18,2026-04-19,False
5,"스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%",55,55,2017-03-26,2025-07-20,False
6,불꽃늑대 퇴장 시 획득하는 경험치 2배,53,53,2017-12-03,2024-08-18,True
7,소울 조각 사용 시 위대한 소울 획득 확률 5배,53,53,2017-04-23,2026-05-17,False
8,몬스터 파크 클리어 경험치 추가 250%,47,47,2023-12-24,2026-05-24,False
9,10성 이하에서 스타포스 강화 성공 시 1+1 강화,42,42,2018-02-18,2026-01-11,False


## 3단계 파서 보정 확인


In [6]:
no_event = tidy_prenorm[tidy_prenorm['no_event_week']].copy()
empty_unknown = tidy_prenorm[
    (~tidy_prenorm['no_event_week']) & tidy_prenorm['reward_raw'].isna()
].copy()
future_empty = empty_unknown[empty_unknown['is_future']].copy()
past_empty = empty_unknown[~empty_unknown['is_future']].copy()

print('parsed_by_type 단종 고유 보상 수:', parsed_by_type.loc[parsed_by_type['is_discontinued'], 'reward_raw'].nunique())
print('no_event_week 행 수:', len(no_event))
print('no_event_week 중 미래 행 수:', int(no_event['is_future'].sum()))
print('미래 빈 블록 수:', len(future_empty))
print('센티넬 없는 과거 빈 블록 수:', len(past_empty))
print('최소 날짜:', tidy_prenorm['date'].min().date().isoformat())
print('최대 날짜:', tidy_prenorm['date'].max().date().isoformat())
print('고유 날짜 수:', tidy_prenorm['date'].nunique())
display(past_empty[['date', 'event_context', 'is_major_event', 'is_future', 'no_event_week']])


parsed_by_type 단종 고유 보상 수: 71
no_event_week 행 수: 30
no_event_week 중 미래 행 수: 0
미래 빈 블록 수: 12
센티넬 없는 과거 빈 블록 수: 0
최소 날짜: 2017-03-12
최대 날짜: 2026-08-30
고유 날짜 수: 501


,date,event_context,is_major_event,is_future,no_event_week
